# 01. Kaggle 3종 breakpoint 찾기

EuroSAT, FGVC-Aircraft, DTD에서 2SFS Stage 1을 학습하며 Base/Novel 조화평균이 가장 높은 지점을 찾는다. 학습에는 Base class의 benchmark seed-1 few-shot만 사용하고, breakpoint 선택에는 raw official validation 전체를 사용한다. 이 노트북에서는 test label을 읽지 않는다.

## 0. 프로젝트 내려받기

새 Jupyter 작업 공간에서 시작한다면 아래 셀을 한 번 실행한다. 이미 repository 안에서 이 노트북을 열었다면 건너뛴다. Clone 후 `uv sync`로 환경을 준비하고 `.venv` kernel을 선택한다.

In [ ]:
!git clone -b ky --depth 1 https://github.com/Twave1717/ELLab-MLStudy.git
%cd ELLab-MLStudy

## 실행 전 준비

- 프로젝트 가상환경을 사용하는 Jupyter kernel을 선택한다.
- `kaggle/public` manifest와 연결된 원본 이미지가 있어야 한다.
- 결과는 데이터셋별 JSON으로 저장되며, 같은 설정의 완료 결과는 다시 계산하지 않는다.
- GPU가 한 장이므로 세 데이터셋을 순차 실행한다. Seed 설정과 CUDA 메모리가 동시에 섞이지 않아 결과를 재현하기 쉽다.

## 1. 환경과 모듈

노트북에는 breakpoint 선택 흐름을 직접 작성하고, manifest 로딩·학습 루프·평가·JSON 저장처럼 반복되는 구현만 `notebook/utils`에서 가져온다.

In [ ]:
from pathlib import Path
import sys

import torch

ROOT = next(
    path
    for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "pyproject.toml").is_file()
)
sys.path[:0] = [str(ROOT / "notebook"), str(ROOT)]

from utils import data, experiment, results

## 2. 실험 설정

16-shot LayerNorm을 shot당 300 step, 즉 총 4,800 step 학습한다. 매 10 step마다 validation 전체를 평가한다. 저장된 결과를 무시하고 다시 실행하려면 `RESTART = True`로 바꾼다.

In [ ]:
DATASETS = data.DATASETS
SHOTS, PEFT = 16, "ln"
BATCH_SIZE, LR = 32, 2e-4
STEPS_PER_SHOT, PROBE_EVERY_STEPS = 300, 10
SEED, RESTART = 2026, False

CONFIGS = {
    name: experiment.ExperimentConfig(
        repo_root=ROOT,
        datasets=(name,),
        shots=SHOTS,
        peft=PEFT,
        batch_size=BATCH_SIZE,
        lr=LR,
        steps_per_shot=STEPS_PER_SHOT,
        probe_every_steps=PROBE_EVERY_STEPS,
        seed=SEED,
    )
    for name in DATASETS
}

## 3. 입력 데이터 점검

`train`은 Base class당 16장으로 제한되지만 `val`은 few-shot으로 줄이지 않는다. 아래 셀은 class 수, 학습 이미지 수, raw official validation 수만 확인하며 모델 학습이나 test 평가는 수행하지 않는다.

In [ ]:
input_config = experiment.ExperimentConfig(
    repo_root=ROOT, datasets=DATASETS, shots=SHOTS, peft=PEFT
)
print(results.input_summary(input_config))

dataset       | classes | B cls | N cls | train | val 
--------------+---------+-------+-------+-------+-----
eurosat       | 10      | 5     | 5     | 80    | 5400
fgvc_aircraft | 100     | 50    | 50    | 800   | 3333
dtd           | 47      | 24    | 23    | 384   | 1128


## 4. 핵심 함수: breakpoint 측정

핵심은 다음 세 단계다.

1. Base few-shot으로 `TwoStageCLIP`의 Stage 1을 학습한다.
2. 학습 전(step 0)과 이후 10 step마다 Base/Novel validation 정확도 및 `H = 2BN / (B + N)`을 기록한다.
3. H가 최대인 **최초** step을 breakpoint로 선택한다. Python의 `max`는 동점일 때 먼저 등장한 항목을 유지한다.

모델 준비, minibatch 학습, 전체 validation 평가는 검증된 util을 사용하고 선택 규칙은 아래에서 명시적으로 보여준다.

In [ ]:
def find_breakpoint(config, dataset):
    run = experiment.prepare_breakpoint(config, dataset)
    validation, total_steps, records = run.validation, config.total_steps, []

    print(
        f"{dataset}: validation B={len(validation[0])}, "
        f"N={len(validation[1])}, total={sum(map(len, validation))}"
    )

    def probe(step):
        run.method.eval()
        with torch.inference_mode():
            classifiers = (
                run.method.encode_text(),
                run.method.encode_classnames(validation[1].classes),
            )
        metrics = experiment.evaluate(
            run.method, validation, classifiers, config, run.device
        )
        records.append({"step": step, "ratio": step / total_steps, **metrics})
        print(
            f"{dataset} probe [{step}/{total_steps}] "
            f"B={metrics['base_accuracy']:.4f} "
            f"N={metrics['novel_accuracy']:.4f} "
            f"H={metrics['harmonic_mean']:.4f}"
        )
        run.method.train()

    probe(0)
    run.method.train()
    experiment.train_steps(
        run.method.stage_one_logits,
        run.parameters,
        run.train_loader,
        total_steps,
        config,
        run.device,
        f"{dataset} breakpoint stage1",
        probe,
    )

    best = max(records, key=lambda row: row["harmonic_mean"])
    selected = {
        "auto_step": best["step"],
        "auto_ratio": best["step"] / total_steps,
        "auto_harmonic_mean": best["harmonic_mean"],
    }
    result = experiment.build_breakpoint_result(
        config, dataset, validation, records, selected
    )
    device = run.device
    del run
    if device.type == "cuda":
        torch.cuda.empty_cache()
    return result

## 5. 결과 저장과 재사용

각 데이터셋은 `results/logs/kaggle_breakpoint/breakpoints_<dataset>_ln_16shot.json`에 독립적으로 저장한다. 파일의 설정 signature가 현재 설정과 다르면 중단하고, 일치하며 완료된 결과가 있으면 그대로 재사용한다.

In [ ]:
def search_breakpoint(config, dataset, restart=False):
    path, payload = experiment.open_breakpoint_checkpoint(
        config, dataset, restart
    )
    if dataset in payload["datasets"]:
        print(f"{dataset}: reuse {path.relative_to(ROOT)}")
        return payload

    payload["datasets"][dataset] = find_breakpoint(config, dataset)
    experiment.save_json(path, payload)
    print(f"{dataset}: saved {path.relative_to(ROOT)}")
    return payload

## 6. 세 데이터셋 실행

한 GPU에서 모델 세 개를 동시에 올리지 않고 EuroSAT → FGVC-Aircraft → DTD 순서로 실행한다. 셀이 끝나면 `runs`에 세 JSON payload가 남고, 학습 로그도 notebook output에 보존할 수 있다.

In [ ]:
runs = {}
for dataset in DATASETS:
    runs[dataset] = search_breakpoint(
        CONFIGS[dataset], dataset, restart=RESTART
    )

eurosat: reuse results/logs/kaggle_breakpoint/breakpoints_eurosat_ln_16shot.json
fgvc_aircraft: reuse results/logs/kaggle_breakpoint/breakpoints_fgvc_aircraft_ln_16shot.json
dtd: reuse results/logs/kaggle_breakpoint/breakpoints_dtd_ln_16shot.json


## 7. 선택 결과 확인

`auto step`과 `auto ratio`는 validation H가 처음 최대가 된 위치다. 아래 output은 저장된 세 JSON을 다시 읽어 만든 실제 결과다.

In [ ]:
print(results.breakpoint_summary(PEFT, SHOTS))

dataset       | val  | auto step | auto ratio | best H | selected | source   
--------------+------+-----------+------------+--------+----------+----------
eurosat       | 5400 | 80        | 0.0167     | 0.8229 | 0.0167   | automatic
fgvc_aircraft | 3333 | 1430      | 0.2979     | 0.4035 | 0.2979   | automatic
dtd           | 1128 | 3340      | 0.6958     | 0.7060 | 0.6958   | automatic


## 8. 다음 단계

- EuroSAT은 80/4,800 = 1.67%, FGVC-Aircraft는 1,430/4,800 = 29.79%, DTD는 3,340/4,800 = 69.58%를 선택했다.
- 이 값은 validation에서 선택한 비율이며 아직 test 성능을 의미하지 않는다.
- `02_compare_breakpoint_ratios.ipynb`에서 논문의 고정 비율 60%와 선택 비율을 같은 test split에서 비교한다.